# Step 1 -- Reproduce idiom steering (IdioSteer)

Goal: build the idiom-derived mean-difference steering vector from IdioLink and
confirm the core Steering Idioms result on `data/idiom_eval_benchmark_draft.csv`
-- steering should push continuations from figurative toward literal readings
while preserving fluency.

All reusable logic (model loading, activation collection, the steering hook,
the resumable eval loop, the judge) lives in `src/steering_pipeline.py`. This
notebook is a thin runner over it. GPU runtime required (Colab T4 is enough
for Llama-3.2-3B + Qwen2.5-3B-Instruct, run sequentially, not concurrently).

## Setup: clone repo, install deps, import the shared pipeline

In [ ]:
import os

REPO_DIR = "beyond-idiom-steering"
if not os.path.isdir(REPO_DIR):
    # Running fresh in Colab: clone the repo. If you already have the repo
    # mounted (e.g. via Drive or `%cd`), skip this cell and just make sure
    # your working directory is the repo root.
    !git clone https://github.com/Itamarvs/beyond-idiom-steering.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, ".")


In [ ]:
from src.steering_pipeline import *
import pandas as pd


## 1. Load the base generation model

In [ ]:
model, tokenizer, device = load_generation_model()
print("device:", device, "| model:", MODEL_NAME)


## 2. Load the IdioLink training pool

Combines the `indexes` and `queries` configs (the paper's stated 80
sentences/idiom pool only comes out right with both), keeps idiomatic/literal
rows, and applies the paper's exact-surface-form filter.

In [ ]:
pool_df = load_idiolink_pool()
print("idioms:", pool_df["idiom"].nunique(), "| pool size:", len(pool_df))
print(pool_df["label"].value_counts())
pool_df.head()


## 3. Build and save the mean-difference steering vector (Eq. 1)

In [ ]:
v_md, s_md = build_steering_vector(pool_df, model, tokenizer, device, source_layer=SOURCE_LAYER)
print("raw mean-difference norm (s_md):", s_md)
print("unit vector shape:", v_md.shape)

save_steering_vector("results/steering_vector_llama3.2-3b.pkl", v_md, s_md, SOURCE_LAYER, MODEL_NAME)
print("saved results/steering_vector_llama3.2-3b.pkl")


## 4. Sanity check on a few idiom prefixes

In [ ]:
eval_df = pd.read_csv("data/idiom_eval_benchmark_draft.csv")

sanity_idioms = ["break the ice", "spill the beans", "see red"]
sanity_rows = eval_df[eval_df["idiom"].isin(sanity_idioms) & (eval_df["variant_id"] == 1)]

for _, row in sanity_rows.iterrows():
    prefix = row["prefix"]
    print("=" * 80)
    print(f"IDIOM: {row['idiom']}")
    unsteered = steered_generate_batch(model, tokenizer, device, prefix, v_md, s_md, alpha_factor=0.0, n_samples=1)[0]
    print(f"UNSTEERED : {prefix}{unsteered}")
    steered_lit = steered_generate_batch(model, tokenizer, device, prefix, v_md, s_md, alpha_factor=-4.78, n_samples=1)[0]
    print(f"STEERED(-4.78, literal push): {prefix}{steered_lit}")
    print()


## 5. Full evaluation sweep (resumable -- safe to rerun after a disconnect)

In [ ]:
idiom_results = run_generation_eval(
    model, tokenizer, device, eval_df, v_md, s_md,
    out_csv="results/raw_idiom_results.csv",
)
print(f"total rows: {len(idiom_results)}")


## 6. Free the generation model before loading the judge

Tight on a single T4: the 3B generation model plus a 4-bit-quantized 14B
judge should both fit (~6GB + ~9GB), but only if the generation model isn't
still resident.

In [ ]:
del model
torch.cuda.empty_cache()
print("Generation model freed.")


## 7. LLM-as-judge labeling (Qwen2.5-14B-Instruct, 4-bit)

The Steering Idioms paper (Appendix J) used **Gemma-4-31B-it** as its judge,
selected by benchmarking several candidates (GPT-4o, Claude Sonnet 3.5,
Llama-3.3-70B-Instruct, Qwen3, ...) against a 280-item human-annotated gold
set: 90.0% accuracy, Cohen's kappa=0.821 vs. gold (human-human kappa=0.867).
A 31B judge doesn't fit a free-tier Colab GPU, so this uses
**Qwen2.5-14B-Instruct in 4-bit** -- the largest judge that reliably fits,
and a meaningful capacity step up from the original Qwen2.5-3B-Instruct
judge (see `results/labeled_idiom_results_qwen3b_baseline.csv`), which
produced a flat literal rate across the whole alpha grid (~0.30-0.41, no
clear steering effect) -- unlike the paper's own reported result at this
*exact* calibrated config (`L_s=14, L_i=2, alpha_factor` grid), which shows
baseline 0.12 -> 0.49 at alpha_factor=-4.78. That mismatch, at an otherwise
identical config, points at the judge as the likely weak link, not the
steering vector -- this cell is the test of that hypothesis.

In [ ]:
judge_model, judge_tokenizer, device = load_judge_model()


In [ ]:
labeled_df = run_judge_eval(
    judge_model, judge_tokenizer, device,
    in_csv="results/raw_idiom_results.csv",
    out_csv="results/labeled_idiom_results.csv",
)
summarize_labels(labeled_df, group_cols=["alpha_factor"])


## 8. Commit results (optional -- run manually, review `git status` first)

Only run this if you actually want to push from the Colab session. Safer
default: download `results/*.csv` and the `.pkl` and commit locally.

In [ ]:
# !git add results/
# !git commit -m "Step 1: steering vector + idiom eval results"
# !git push
